# 04. 学習と評価 — 03 の特徴量で 4 モデルを学習し、本番のスコアを再現する

`03_feature_engineering.ipynb` で確かめた特徴量の作り方を読み込み、**4 モデル(LightGBM / XGBoost / CatBoost / RealMLP)を
本番と同じ設定で学習して、OOF AUC を出す**ノートブック。上から順に実行すれば、本番のスコアが再現できる。

## このノートブックの流れ

| 章 | 内容 | 実行時間の目安 |
|---|---|---|
| 1 | 準備(読み込み・fold 分割・本番の予測との照合の仕組み) | 数秒 |
| 2 | LightGBM(46 列) | 約 4 分 |
| 3 | XGBoost(69 列) | 約 8 分 |
| 4 | CatBoost(43 列) | 約 22 分 |
| 5 | RealMLP(38 列) | 約 41 分 |
| 6 | アンサンブルとまとめ(本番のスコアとの照合、最終提出) | 数秒 |
| 7 | (補足)特徴量を1種類ずつ足した比較 — LightGBM の縮小版 | 約 6 分 |

**全体で約 80 分**

- 特徴量は、03 で確かめた作り方(`src/03_feature_engineering_<model>.py` の関数)を読み込んで作る。
  Target Encoding は fold ごとに作り直す必要があるので、5 fold 分をここで作る
- 各モデルの予測は、本番の予測(`oof/oof_<model>.npy`)と行ごとに照合する。**本番の成果物は上書きしない**
- test の予測と提出ファイルは作らない(本番は `src/04_train_and_evaluate_<model>.py` が作る。6 章にコマンド)

## `src/` を読み込む理由と、ノートブックに移さない理由

特徴量を作る関数(`src/03_feature_engineering_<model>.py`)と RealMLP のモデル本体(`src/04_train_and_evaluate_realmlp.py`)は、
**ノートブックに書き写さず `src/` から読み込む**。ファイル名が数字で始まり、そのままでは `import` と書けないので `importlib` を使っている。

ノートブックだけで完結させる方法(03 から列名や特徴量の値を渡す)も検討したが、次のデメリットがあるので採らない。

| デメリット | 内容 |
|---|---|
| 正本が2つになる | 本番(`src/` の .py)とノートブックに同じ処理が並ぶと、片方だけ直したときに黙ってずれる。本番のスコアを再現するという、このノートブックの目的が崩れる |
| 列名だけでは足りない | Target Encoding の値は fold ごとに変わるので、04 は 5 fold 分を作り直す必要がある。03 から列名を受け取っても、値を作る関数は結局 `src/` から読むことになる |
| 値まで渡すと重い | 03 で 4 モデル × 5 fold の特徴量を保存すると、概算で 1〜2GB になり、03 の実行時間も延びる |
| ノートブックのセルに依存する | 03 の 4-4・5-9 の一覧をセルから切り出して読むと、セルの並びや書き方を変えたときに壊れる |
| `docs/features_*.json` は流用できない | 本番の `--dump-features` を実行するたびに上書きされる |

> **クローン・持ち出しの際は、`notebooks/` だけでなく `src/` も必ず含めること。** `src/` がないと、1 章の準備のあとで読み込みに失敗する。
> また `data/` と `oof/` はリポジトリに含めていない。クローン先では `data/` を Kaggle から取得し、本番の照合に使う `oof/oof_<model>.npy` は `src/04_train_and_evaluate_<model>.py` を先に実行して作る(6 章にコマンド)。

## 05(Hyperparameter Tuning)との関係と、パラメータを値で書いている理由

**05 の位置づけ**: `src/05_hyperparameter_tuning.py` は学習コードを持たない。`src/04_train_and_evaluate_<model>.py` を
引数を変えて1本ずつ呼び、OOF AUC を `docs/hyperparameter_tuning_results.csv` に記録するだけの工程。

```
05_hyperparameter_tuning.py ──(引数を変えて呼ぶ)──→ 04_train_and_evaluate_<model>.py ──→ OOF AUC
        └──→ docs/hyperparameter_tuning_results.csv に記録 → 人が DeLong 検定で採否を判断
```

- 呼ぶ向きは **05 → 04 だけ**。05 の結果が 04 に自動で反映される仕組みはない
- 05 で実行した 18 試行(記録は延べ 21 本・合計約 4.2 時間。`num_leaves` の 3 本は2回実行)はすべて誤差か悪化で、**採用はゼロ**。本番のパラメータは探索前の値のまま
- 本番のパラメータの**正本は 05 の `FIXED`**(本番の実行コマンドのうち、探索しない固定部分)

**このノートブックのパラメータは、実行済みの結果から決まった値を書き写したもの**(05 や csv から読み込んではいない)。

| 値だけを書いている理由 | 内容 |
|---|---|
| 探索をやり直さない | 05 の探索は合計約 4.2 時間かかり、結果は確定している。ここで探し直す意味がない |
| 読み込む元がない | csv にあるのは試した値の記録で、「採用した値」は書かれていない(採用ゼロのため) |
| 設定が見える | 学習セルを読めば、何を使ったかがそのまま分かる |
| 正本とのずれは照合で防ぐ | 6 章で 05 の `FIXED` と値を突き合わせ、ずれていれば止まる。加えて、予測が本番と完全一致することも確かめている |

RealMLP は 05 の探索対象外。設定は `src/04_train_and_evaluate_realmlp.py` の `make_config()` をそのまま使う。

## 1. 準備

fold 分割は全モデル共通(`StratifiedKFold(5, shuffle=True, random_state=42)`)。
`check()` は、ノートブックで作った予測を本番の予測と比べ、AUC と値の最大差を表示する。

In [1]:
import os, sys, time, importlib, warnings
# リポジトリルートを作業ディレクトリにして、data/ などの相対パスを揃える
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.path.abspath("src"))

import numpy as np
import pandas as pd
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

TARGET = "Will_Buy_EV"
N_JOBS = 7                                   # 本番と同じスレッド数(8 コアのうち 7)

train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
y = (train[TARGET] == "Yes").astype(int).to_numpy()
FOLDS = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(train, y))

NB_OOF, TIMES = {}, {}                        # モデル名 -> ノートブックで作った OOF 予測 / 所要時間


def check(name, oof, seconds):
    # ノートブックの予測を、本番の予測(oof/oof_<name>.npy)と照合する
    NB_OOF[name], TIMES[name] = oof, seconds
    prod = np.load(f"oof/oof_{name}.npy")
    print(f"{name}: ノートブック {roc_auc_score(y, oof):.6f} / 本番 {roc_auc_score(y, prod):.6f}"
          f" / 予測の最大差 {np.abs(oof - prod).max():.1e}({seconds / 60:.1f} 分)")


# 絶対パスは出力しない(公開リポジトリに個人のディレクトリ構成を残さないため)
print(f"data/ を検出: {os.path.isdir('data')} / train {train.shape} / test {test.shape} / 購入率 {y.mean():.3f}")

data/ を検出: True / train (668665, 15) / test (286571, 14) / 購入率 0.175


## 2. LightGBM

**特徴量**: 03 の 4-1 と同じ 46 列(生の 13 列 + digit 7 + Count 2 + Target Encoding 24)。
本番と同じ呼び方で、いったん全列にエンコーディングをかけてから、重複(`dedup_columns()`)と
不要なエンコーディング(`lean_columns()`)を外す(03 のノートブックは必要な列だけを作る書き方だが、結果は同じ。03 の 9 章で確認済み)。

**モデル**: 深さ 5・1 本の木が見る列を 30% に絞る・ビン数 1024・学習率 0.03。検証 fold の AUC が 200 本改善しなければ止める(early stopping)。

In [2]:
fe_lgbm = importlib.import_module("03_feature_engineering_lgbm")

# fold によらない部分: 生の列(カテゴリは category 型)+ digit + Count
train_c, test_c = fe_lgbm.make_categorical(train, test)
keys_tr, keys_te = fe_lgbm.make_key_frame(train, test)
keys_tr, keys_te = fe_lgbm.add_smooth_keys(keys_tr, train), fe_lgbm.add_smooth_keys(keys_te, test)
digit_tr = fe_lgbm.add_digit_features(train)
cnt_tr, _ = fe_lgbm.count_encode(keys_tr, keys_te, fe_lgbm.single_keys("all"))
X_static = pd.concat([train_c[fe_lgbm.NUMERIC_COLS + fe_lgbm.CATEGORICAL_COLS], digit_tr, cnt_tr], axis=1)

# Target Encoding のキー: 生の 13 列 + Smooth Keys 4 本 + 組み合わせ 1 組(自宅充電 × 自宅スタンド数)
LGBM_TE_KEYS = (fe_lgbm.single_keys("all") + fe_lgbm.SMOOTH_KEYS
                + [("Home_Charging_Possible", "Charging_Stations_Near_Home")])
LGBM_DROP = set(fe_lgbm.dedup_columns() + fe_lgbm.lean_columns())
print(f"fold によらない列 {X_static.shape[1]} 本 / Target Encoding のキー {len(LGBM_TE_KEYS)} 本 / 外す候補 {len(LGBM_DROP)} 本")

fold によらない列 41 本 / Target Encoding のキー 18 本 / 外す候補 49 本


In [3]:
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
import warnings

# LightGBM の「eval_set 引数は将来廃止予定」という通知は動作に影響しない(本番スクリプトでも同じ通知が出る)ので表示しない
warnings.filterwarnings("ignore", message="The argument 'eval_set' is deprecated")

LGBM_PARAMS = dict(random_state=42, verbosity=-1, n_estimators=8000, learning_rate=0.03, num_leaves=31,
                   max_bin=1024, n_jobs=N_JOBS, colsample_bytree=0.3, max_depth=5)
LGBM_EARLY_STOP = 200                        # 検証 fold の AUC が 200 本改善しなければ止める

oof, t0 = np.zeros(len(train)), time.time()
for fold, (tr_idx, va_idx) in enumerate(FOLDS):
    # Target Encoding は fold ごとに作る(学習行は Out-of-Fold、検証行は学習行すべてで計算)
    te_tr, (te_va,) = fe_lgbm.target_encode_fold(keys_tr.iloc[tr_idx], y[tr_idx], [keys_tr.iloc[va_idx]],
                                                 LGBM_TE_KEYS, smooth=["auto", 10.0, 100.0], n_inner=5, seed=42)
    X_tr = pd.concat([X_static.iloc[tr_idx].reset_index(drop=True), te_tr.reset_index(drop=True)], axis=1)
    X_va = pd.concat([X_static.iloc[va_idx].reset_index(drop=True), te_va.reset_index(drop=True)], axis=1)
    X_tr, X_va = (f.drop(columns=[c for c in f.columns if c in LGBM_DROP]) for f in (X_tr, X_va))

    model = LGBMClassifier(**LGBM_PARAMS)
    model.fit(X_tr, y[tr_idx], eval_set=[(X_va, y[va_idx])], eval_metric="auc",
              callbacks=[early_stopping(LGBM_EARLY_STOP, verbose=False), log_evaluation(0)])
    oof[va_idx] = model.predict_proba(X_va)[:, 1]
    print(f"fold {fold}: AUC {roc_auc_score(y[va_idx], oof[va_idx]):.5f} / 木 {model.best_iteration_} 本 / {X_tr.shape[1]} 列")

check("lgbm", oof, time.time() - t0)
del X_static, keys_tr, keys_te, train_c, test_c

fold 0: AUC 0.94517 / 木 798 本 / 46 列


fold 1: AUC 0.94571 / 木 715 本 / 46 列


fold 2: AUC 0.94705 / 木 1049 本 / 46 列


fold 3: AUC 0.94643 / 木 874 本 / 46 列


fold 4: AUC 0.94620 / 木 785 本 / 46 列


lgbm: ノートブック 0.946109 / 本番 0.946109 / 予測の最大差 3.6e-13(2.5 分)


## 3. XGBoost

**特徴量**: 03 の 4-2 と同じ 69 列。カテゴリ列は整数コード(ordinal)で渡し、Count は全 13 列に作る。
Target Encoding のキーは Smooth Keys 4 本(年収 /100・/1000・/10000、通勤距離)+ 生の 13 列で、重複(`dedup_columns()`)を外す。

**モデル**: LightGBM と同じ考え方(深さ 5・列 30%・ビン数 1024・学習率 0.03・early stopping 200)。

In [4]:
fe_xgb = importlib.import_module("03_feature_engineering_xgb")
XA = fe_xgb.NUMERIC_COLS + fe_xgb.CATEGORICAL_COLS

# fold によらない部分: 生の列 + digit + Count(全 13 列)→ カテゴリ列を整数コードに
X_x, X_x_te = train[XA].copy(), test[XA].copy()
X_x, X_x_te = fe_xgb.add_digit_features(X_x, X_x_te, train, test)
X_x, X_x_te = fe_xgb.add_count_encoding(X_x, X_x_te, train, test, XA)
X_x, X_x_te = fe_xgb.as_ordinal(X_x, X_x_te, fe_xgb.CATEGORICAL_COLS)

# Target Encoding のキーを train + test で整数コードにそろえる
sk_tr, sk_te = fe_xgb.make_smooth_keys(train, test)
XGB_TE_COLS = list(sk_tr.columns) + XA
codes_tr, codes_te, ncats = fe_xgb.prepare_te_codes(pd.concat([sk_tr, train[XA]], axis=1),
                                                    pd.concat([sk_te, test[XA]], axis=1), XGB_TE_COLS)
XGB_DROP = set(fe_xgb.dedup_columns())
print(f"fold によらない列 {X_x.shape[1]} 本 / Target Encoding のキー {len(XGB_TE_COLS)} 本")

fold によらない列 42 本 / Target Encoding のキー 17 本


In [5]:
from xgboost import XGBClassifier

XGB_PARAMS = dict(random_state=42, tree_method="hist", n_jobs=N_JOBS, max_bin=1024, colsample_bytree=0.3,
                  max_depth=5, n_estimators=8000, learning_rate=0.03, early_stopping_rounds=200, eval_metric="auc")
y_s = pd.Series(y)

oof, t0 = np.zeros(len(train)), time.time()
for fold, (tr_idx, va_idx) in enumerate(FOLDS):
    te_tr, te_va, _ = fe_xgb.fit_apply_te_cv_nested_multi(codes_tr, codes_te, ncats, y_s, XGB_TE_COLS,
                                                          tr_idx, va_idx, ("auto", 10.0, 100.0), 1)
    X_tr = pd.concat([X_x.iloc[tr_idx].reset_index(drop=True), te_tr], axis=1)
    X_va = pd.concat([X_x.iloc[va_idx].reset_index(drop=True), te_va], axis=1)
    X_tr, X_va = (f.drop(columns=[c for c in f.columns if c in XGB_DROP]) for f in (X_tr, X_va))

    model = XGBClassifier(**XGB_PARAMS)
    model.fit(X_tr, y_s.iloc[tr_idx], eval_set=[(X_va, y_s.iloc[va_idx])], verbose=False)
    oof[va_idx] = model.predict_proba(X_va)[:, 1]
    print(f"fold {fold}: AUC {roc_auc_score(y[va_idx], oof[va_idx]):.5f} / 木 {model.best_iteration + 1} 本 / {X_tr.shape[1]} 列")

check("xgb", oof, time.time() - t0)
del X_x, X_x_te, codes_tr, codes_te

fold 0: AUC 0.94512 / 木 847 本 / 69 列


fold 1: AUC 0.94576 / 木 1172 本 / 69 列


fold 2: AUC 0.94695 / 木 941 本 / 69 列


fold 3: AUC 0.94642 / 木 1043 本 / 69 列


fold 4: AUC 0.94619 / 木 1108 本 / 69 列


xgb: ノートブック 0.946087 / 本番 0.946087 / 予測の最大差 0.0e+00(7.3 分)


## 4. CatBoost

**特徴量**: 03 の 4-3 と同じ 43 列。値の種類（ユニーク値）が少ない数値列を文字列にして cat_features に渡す(catify)。
Target Encoding は平滑化 10 / 20 / 100 で作り、重複(`dedup_columns()`)と不要なエンコーディング(`lean_columns()`)を外す。

**モデル**: 1000 本・学習率 0.06・軽量設定(Plain boosting、ビン数 64)。年収・通勤距離とその Target Encoding の列だけビン数を 1024 にする。
early stopping は使わない。

In [6]:
fe_cb = importlib.import_module("03_feature_engineering_catboost")
CB_BASE = fe_cb.NUMERIC_COLS + fe_cb.CATEGORICAL_COLS

# fold によらない部分: 生の列 + digit(catify の前に作る)+ Smooth Keys(Target Encoding のキー専用)→ catify
X_c, X_c_te = train[CB_BASE].copy(), test[CB_BASE].copy()
cb_features = list(CB_BASE)
digit_cols = fe_cb.add_digits(X_c)
fe_cb.add_digits(X_c_te)
cb_features += fe_cb.drop_constant([X_c, X_c_te], digit_cols)
sk_cols = fe_cb.add_smooth_keys(X_c)
fe_cb.add_smooth_keys(X_c_te)
fe_cb.cast_to_str([X_c, X_c_te], fe_cb.LOWCARD_NUM_COLS)
CB_CATS = fe_cb.CATEGORICAL_COLS + fe_cb.LOWCARD_NUM_COLS
CB_TE_COLS = list(dict.fromkeys(CB_BASE + sk_cols))
X_c = X_c[list(dict.fromkeys(cb_features + sk_cols + CB_TE_COLS))]
X_c_small_te = X_c_te[X_c.columns].iloc[:1]          # test の予測は作らないので、Target Encoding の受け皿は 1 行で足りる
CB_DROP = set(fe_cb.dedup_columns() + fe_cb.lean_columns())
print(f"fold によらない列 {len(cb_features)} 本 / Target Encoding のキー {len(CB_TE_COLS)} 本 / cat_features {len(CB_CATS)} 本")

fold によらない列 29 本 / Target Encoding のキー 17 本 / cat_features 11 本


In [7]:
from catboost import CatBoostClassifier

CB_PARAMS = dict(random_state=42, verbose=False, allow_writing_files=False, iterations=1000, learning_rate=0.06,
                 boosting_type="Plain", max_ctr_complexity=1, border_count=64, thread_count=N_JOBS)
# 年収・通勤距離とその Target Encoding の列だけ、ビン数を 1024 にする
CB_HC_BORDER = 1024
HC_TARGETS = set(fe_cb.HIGHCARD_NUM_COLS) | {f"te{t}_{c}" for c in fe_cb.HIGHCARD_NUM_COLS for t in ("", "10", "20", "100")}

oof, t0 = np.zeros(len(train)), time.time()
for fold, (tr_idx, va_idx) in enumerate(FOLDS):
    X_tr, X_va, X_te_dummy = X_c.iloc[tr_idx].copy(), X_c.iloc[va_idx].copy(), X_c_small_te.copy()
    new_te = fe_cb.target_encode(X_tr, X_va, X_te_dummy, y[tr_idx], CB_TE_COLS,
                                 smooth=20.0, smooths=[10.0, 20.0, 100.0])
    feats = [c for c in cb_features + new_te if c not in CB_DROP]
    cats = [c for c in CB_CATS if c in feats]
    params = dict(CB_PARAMS, per_float_feature_quantization=[
        f"{i}:border_count={CB_HC_BORDER}" for i, name in enumerate(feats) if name in HC_TARGETS and name not in cats])

    model = CatBoostClassifier(**params)
    model.fit(X_tr[feats], y[tr_idx], cat_features=cats)
    oof[va_idx] = model.predict_proba(X_va[feats])[:, 1]
    print(f"fold {fold}: AUC {roc_auc_score(y[va_idx], oof[va_idx]):.5f} / {len(feats)} 列({(time.time() - t0) / 60:.1f} 分)")

check("catboost", oof, time.time() - t0)
del X_c, X_c_te

fold 0: AUC 0.94508 / 43 列(4.1 分)


fold 1: AUC 0.94563 / 43 列(8.3 分)


fold 2: AUC 0.94675 / 43 列(12.5 分)


fold 3: AUC 0.94613 / 43 列(16.6 分)


fold 4: AUC 0.94604 / 43 列(20.8 分)


catboost: ノートブック 0.945924 / 本番 0.945924 / 予測の最大差 0.0e+00(20.8 分)


## 5. RealMLP

**特徴量**: 03 の 5 章と同じ 38 列。組み合わせキーは 3 組(年収 × 航続距離の不安、年齢 × 航続距離の不安、自宅充電 × 自宅スタンド数)で、
その Target Encoding は fold 内で sklearn の `TargetEncoder` を使う。

**モデル**: RealMLP-TD(埋め込み + 3 層 256 ユニット、8 個の内部アンサンブル)を 2 エポック。モデル本体の定義は大きいので、
`src/04_train_and_evaluate_realmlp.py` から読み込む(本番と同じ設定・同じ seed)。

In [8]:
import torch
mlp = importlib.import_module("04_train_and_evaluate_realmlp")    # モデル本体(RealMLP_TD_Classifier)の定義
fe_mlp = importlib.import_module("03_feature_engineering_realmlp")

torch.set_num_threads(N_JOBS)
mlp.seed_everything(42)
MLP_CFG = mlp.make_config(epochs=2, n_ens=8, train_bs=256, log_every=0)

orig = fe_mlp.load_orig("data/EV_Adoption_and_Range_Anxiety_Dataset.csv")
X_m, X_m_te = train.drop(columns=["id", TARGET]), test.drop(columns=["id"])
m_cat = X_m.select_dtypes(include=["object"]).columns.tolist()
m_num = X_m.select_dtypes(exclude=["object"]).columns.tolist()
HOME = [("Home_Charging_Possible", "Charging_Stations_Near_Home")]    # 3 組目の組み合わせキー
cmap = {}
X_m, new_cat, new_num, combos = fe_mlp.build_features(X_m, m_cat, m_num, cmap, fit=True, orig=orig, extra_combos=HOME)
MLP_CATS = m_cat + new_cat
print(f"特徴量 {X_m.shape[1]} 列(+ fold 内の Target Encoding {len(combos)} 列)/ カテゴリとして渡す列 {len(MLP_CATS)} 本")

特徴量 35 列(+ fold 内の Target Encoding 3 列)/ カテゴリとして渡す列 25 本


In [9]:
from sklearn.preprocessing import TargetEncoder

y_ser = pd.Series(y, name=TARGET)
oof, t0 = np.zeros(len(train)), time.time()
for fold, (tr_idx, va_idx) in enumerate(FOLDS, 1):
    X_tr, X_va = X_m.iloc[tr_idx].copy(), X_m.iloc[va_idx].copy()
    y_tr, y_va = y_ser.iloc[tr_idx], y_ser.iloc[va_idx]
    enc = TargetEncoder(cv=5, smooth="auto", shuffle=True, random_state=42)
    te_names = [f"_{c}TE" for c in combos]
    X_tr[te_names] = enc.fit_transform(X_tr[combos], y_tr)
    X_va[te_names] = enc.transform(X_va[combos])

    mlp.seed_everything(42 + fold)
    model = mlp.RealMLP_TD_Classifier(MLP_CFG)
    model.fit(X_tr, y_tr, X_va, y_va, cat_col_names=MLP_CATS)
    oof[va_idx] = model.best_val_probs_
    print(f"fold {fold}: AUC {roc_auc_score(y_va, oof[va_idx]):.5f} / {X_tr.shape[1]} 列({(time.time() - t0) / 60:.1f} 分)")
    del model

check("realmlp", oof, time.time() - t0)

  model built in 0.1s | params=3,415,033 | cat_dims(max)=21543 sum=40661


  epoch 1/2  score=0.94449  best=0.94449  ls=0.0200 drop=0.0068 [219s]  *


  epoch 2/2  score=0.94497  best=0.94497  ls=0.0000 drop=0.0009 [449s]  *


  -> best score: 0.94497 (epoch 2)


fold 1: AUC 0.94497 / 38 列(7.5 分)


...

  epoch 1/2  score=0.94547  best=0.94547  ls=0.0200 drop=0.0068 [220s]  *


  epoch 2/2  score=0.94596  best=0.94596  ls=0.0000 drop=0.0009 [481s]  *


  -> best score: 0.94596 (epoch 2)


fold 5: AUC 0.94596 / 38 列(38.0 分)


realmlp: ノートブック 0.945897 / 本番 0.945897 / 予測の最大差 0.0e+00(38.0 分)


## 6. アンサンブルとまとめ

本番のアンサンブルは、LightGBM・XGBoost・RealMLP の予測を**順位に直して平均**したもの(各 1/3)。
CatBoost は他の GBDT と予測が同質なので重み 0(`06_ensemble.ipynb`)。

In [10]:
rank = lambda v: rankdata(v) / len(v)
ens_nb = sum(rank(NB_OOF[m]) for m in ("lgbm", "xgb", "realmlp")) / 3
ens_prod = sum(rank(np.load(f"oof/oof_{m}.npy")) for m in ("lgbm", "xgb", "realmlp")) / 3

rows = []
for m in ("lgbm", "xgb", "catboost", "realmlp"):
    prod = np.load(f"oof/oof_{m}.npy")
    rows.append([m, roc_auc_score(y, NB_OOF[m]), roc_auc_score(y, prod), np.abs(NB_OOF[m] - prod).max(), TIMES[m] / 60])
rows.append(["アンサンブル(順位の平均)", roc_auc_score(y, ens_nb), roc_auc_score(y, ens_prod), np.abs(ens_nb - ens_prod).max(), np.nan])
summary = pd.DataFrame(rows, columns=["モデル", "ノートブックの OOF AUC", "本番の OOF AUC", "予測の最大差", "所要時間(分)"])
print(summary.to_string(index=False, float_format=lambda v: f"{v:.6f}" if abs(v) < 1 else f"{v:.1f}"))

          モデル  ノートブックの OOF AUC  本番の OOF AUC   予測の最大差  所要時間(分)
         lgbm         0.946109     0.946109 0.000000      2.5
          xgb         0.946087     0.946087 0.000000      7.3
     catboost         0.945924     0.945924 0.000000     20.8
      realmlp         0.945897     0.945897 0.000000     38.0
アンサンブル(順位の平均)         0.946245     0.946245 0.000000      NaN


### 本番のパラメータとの照合(05 の `FIXED`)

このノートブックに書いたパラメータが、正本(`src/05_hyperparameter_tuning.py` の `FIXED`)と同じかを確かめる。
**ずれていれば、このセルで止まる。**

特徴量の構成(`--patterns` など)はこの照合の対象外。こちらは、予測が本番と完全一致すること(上の表)で確かめている。

In [ ]:
# 05 の FIXED(本番の設定の正本)と、このノートブックで使ったパラメータを突き合わせる
tuning = importlib.import_module("05_hyperparameter_tuning")


def fixed_args(model):
    # FIXED の引数の並びを {フラグ: 値} にする。値を取らないフラグは True、--set-param は key=value をほどく
    args, out, i = tuning.FIXED[model], {}, 0
    while i < len(args):
        flag = args[i]
        has_val = i + 1 < len(args) and not args[i + 1].startswith("--")
        val = args[i + 1] if has_val else True
        if flag == "--set-param":
            flag, val = val.split("=")
        out[flag] = val
        i += 2 if has_val else 1
    return out


# (モデル, FIXED のフラグ, このノートブックで使った値)
PAIRS = [
    ("lgbm", "--learning_rate", LGBM_PARAMS["learning_rate"]),
    ("lgbm", "--n_estimators", LGBM_PARAMS["n_estimators"]),
    ("lgbm", "--early_stopping", LGBM_EARLY_STOP),
    ("lgbm", "--max_depth", LGBM_PARAMS["max_depth"]),
    ("lgbm", "--feature_fraction", LGBM_PARAMS["colsample_bytree"]),
    ("lgbm", "--max_bin", LGBM_PARAMS["max_bin"]),
    ("lgbm", "--n_jobs", LGBM_PARAMS["n_jobs"]),
    ("xgb", "--learning-rate", XGB_PARAMS["learning_rate"]),
    ("xgb", "--n-estimators", XGB_PARAMS["n_estimators"]),
    ("xgb", "--early-stopping", XGB_PARAMS["early_stopping_rounds"]),
    ("xgb", "max_depth", XGB_PARAMS["max_depth"]),
    ("xgb", "colsample_bytree", XGB_PARAMS["colsample_bytree"]),
    ("xgb", "--max-bin", XGB_PARAMS["max_bin"]),
    ("xgb", "--n-jobs", XGB_PARAMS["n_jobs"]),
    ("catboost", "--iters", CB_PARAMS["iterations"]),
    ("catboost", "--lr", CB_PARAMS["learning_rate"]),
    ("catboost", "--fast", CB_PARAMS["boosting_type"] == "Plain" and CB_PARAMS["max_ctr_complexity"] == 1),
    ("catboost", "--border", CB_PARAMS["border_count"]),
    ("catboost", "--hc-border", CB_HC_BORDER),
    ("catboost", "--threads", CB_PARAMS["thread_count"]),
]
rows = []
for model, flag, nb_val in PAIRS:
    fixed = fixed_args(model)[flag]
    same = fixed == nb_val if isinstance(nb_val, bool) else float(fixed) == float(nb_val)
    rows.append([model, flag, fixed, nb_val, "一致" if same else "不一致"])
table = pd.DataFrame(rows, columns=["モデル", "FIXED の引数", "FIXED の値", "ノートブックの値", "判定"])
print(table.to_string(index=False))
assert (table["判定"] == "一致").all(), "05 の FIXED と、このノートブックのパラメータがずれている"
print(f"\n{len(table)} 項目すべて一致")

     モデル          FIXED の引数 FIXED の値 ノートブックの値 判定
    lgbm    --learning_rate     0.03     0.03 一致
    lgbm     --n_estimators     8000     8000 一致
    lgbm   --early_stopping      200      200 一致
    lgbm        --max_depth        5        5 一致
    lgbm --feature_fraction      0.3      0.3 一致
    lgbm          --max_bin     1024     1024 一致
    lgbm           --n_jobs        7        7 一致
     xgb    --learning-rate     0.03     0.03 一致
     xgb     --n-estimators     8000     8000 一致
     xgb   --early-stopping      200      200 一致
     xgb          max_depth        5        5 一致
     xgb   colsample_bytree      0.3      0.3 一致
     xgb          --max-bin     1024     1024 一致
     xgb           --n-jobs        7        7 一致
catboost            --iters     1000     1000 一致
catboost               --lr     0.06     0.06 一致
catboost             --fast     True     True 一致
catboost           --border       64       64 一致
catboost        --hc-border     1024     1024 一致
catboost          --

### 本番の構成(詳しくは `03_feature_engineering.ipynb`)

#### 各モデルが作るエンコーディング

| モデル | digit | Target Encoding | Count | 交互作用(自宅充電 × 自宅スタンド数) | 列数 |
|---|---|---|---|---|---|
| LightGBM | 年収・通勤距離 | 値の種類（ユニーク値）が多い列は平滑化3種、少ない数値列は1種。カテゴリ列には作らない | 年収・通勤距離 | **○**(Target Encoding 1 列) | 46 |
| XGBoost | 保有台数・環境意識以外 | 値の種類（ユニーク値）が少ない11キーは1種、他は3種 | 全13列 | — | 69 |
| CatBoost | 年収・通勤距離・保有台数・環境意識 | 値の種類（ユニーク値）が少ない数値列は1種。カテゴリ列には作らない | — | — | 43 |
| RealMLP | (通勤距離の小数部のみ) | 組み合わせキー3組 | 年収(train で数える) | **○**(組み合わせキー + Target Encoding) | 38 |

#### 最終提出(2 本)

| 提出 | 内容 | CV | Public LB |
|---|---|---|---|
| **D** | 上の構成(現在のコード) | **0.946245**(最高) | 0.94640 |
| 元の構成 | 列の整理と交互作用の採用より前(git のタグ `final-original-20260927`) | 0.946234 | **0.94645**(最高) |

CV の差は有意でないので、Private に近い CV の最高と、Public の最高を1本ずつ選んだ。

#### 本番の実行コマンド(リポジトリのルートで実行)

```bash
uv run src/04_train_and_evaluate_lgbm.py --patterns base,te1,cnt1,digit,sk,te2home --smooths auto,10,100 --dedup --lean \
  --max_bin 1024 --feature_fraction 0.3 --max_depth 5 \
  --folds 5 --learning_rate 0.03 --n_estimators 8000 --early_stopping 200 --n_jobs 7 --save --tag lgbm
uv run src/04_train_and_evaluate_xgb.py --pattern tte_sk_dig --dedup --max-bin 1024 \
  --set-param colsample_bytree=0.3 --set-param max_depth=5 \
  --folds 5 --learning-rate 0.03 --n-estimators 8000 --early-stopping 200 --n-jobs 7 --save --out-suffix ""
uv run src/04_train_and_evaluate_catboost.py --fe te_all,catify,digits,skeys,te3 --dedup --lean \
  --folds 5 --iters 1000 --lr 0.06 --fast --border 64 --hc-border 1024 --threads 7 --save
uv run src/04_train_and_evaluate_realmlp.py --folds 5 --threads 7 --combo-home --tag realmlp
```

## 7. (補足)特徴量を1種類ずつ足した比較 — LightGBM の縮小版

以前の 04 の内容。特徴量を1種類ずつ足しながら、AUC がどう変わるかを LightGBM で確かめる
(2 fold・Target Encoding は平滑化1種の縮小版なので、値は本番より小さく出る)。

### 7-1 結論

| 施策 | 本番での効果 | この比較での実測(7-3) | 本番での判定 | 03 の節 |
|---|---|---|---|---|
| **厳密値 Target Encoding** | **+0.00345**(LightGBM) | B: +0.00177 | **採用**。最大の改善要因 | 3-6 |
| Count Encoding | +0.00083(LightGBM)/ +0.00049(XGBoost) | C: +0.00029 | **採用**(LightGBM は値の種類（ユニーク値）が多い2列だけ) | 3-7 |
| 平滑化3種 + Smooth Keys + digit + ビン数 1024 | +0.00048〜0.00096 | (本ノートブックでは未実施) | **採用** | 3-2 / 3-4 / 3-6 |
| 重複の削除・エンコーディングの絞り込み | ±0(列が 3 分の 1 に減る) | (本ノートブックでは未実施) | **採用**(整理のため) | 3-3 / 8-7 |
| **交互作用: 自宅充電の可否 × 自宅スタンド数** | LightGBM +0.000012 / RealMLP +0.000028(誤差) | E: +0.00002(誤差) | **LightGBM・RealMLP に1組だけ採用** | 3-3 / 5-8 / 8-1 |
| その他の交互作用(全ペア・3列の組み合わせなど) | 無効 | (本ノートブックでは未実施) | 不採用 | 8-3 |
| 四則演算(和・差・比など) | 無効〜悪化 | D: -0.00003 | 不採用 | 8-2 |

**効いたのは「値の細部を失わずにモデルへ渡す」施策**(厳密値 Target Encoding・Count・平滑化3種・digit・ビン数)だった。
「組み合わせる」施策(交互作用・四則演算)は、基本的に効かない。木は2列の条件分岐を自力で表現できるためである。

> **採用した交互作用は1組だけ。** EDA で見つかった唯一の交互作用「自宅充電の可否 × 自宅スタンド数」は、
> LightGBM と RealMLP で CV がわずかに上がった。これを含む構成(D)がアンサンブルの CV 最高(0.946245)だったため、
> 最終提出に残した。**採用基準(+0.00008 かつ z ≥ 3)は満たしていない**ので、「CV 最高を残す」という判断による採用である。

In [11]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier

TARGET = "Will_Buy_EV"
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
y = (train[TARGET] == "Yes").astype(int)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

NUM_COLS = [c for c in train.select_dtypes(include=[np.number]).columns if c != "id"]
CAT_COLS = [c for c in train.columns if c not in NUM_COLS + ["id", TARGET]]
ALL_COLS = NUM_COLS + CAT_COLS

### 7-2 準備と評価の仕組み

#### 7-2-1 なぜ Target Encoding が効くのか(詳しくは 03 の 3-6)

EDA で見たとおり年収は 13,214 種類の値を持つ。木は既定で 255 ビンにまとめるため、
**値ごとに違う購入率を直接は学べない**。Target Encoding はその「値ごとの購入率」を 1 列で渡す。

数値列もビン分割せず **厳密な値のままキーにする**のが要点。

In [12]:
rate = y.groupby(train["Annual_Income_USD"]).agg(["mean", "size"])
print("年収のユニーク値:", len(rate))
print("1値あたりの平均行数:", round(rate["size"].mean(), 1))
rate.head()

年収のユニーク値: 13214
1値あたりの平均行数: 50.6


,mean,size
Annual_Income_USD,,
30000.0,0.044266,61605
31003.0,0.500000,2
38174.0,0.000000,1
38209.0,0.000000,1
38250.0,0.000000,1


#### 7-2-2 リーク対策 — Out-of-Fold Target Encoding(詳しくは 03 の 3-6)

Target Encoding は目的変数を使うため、作り方を誤ると学習データの答えが漏れる。ここでは二重に防ぐ。

1. 外側 fold の学習データ内だけで統計を計算する
2. **学習行には内側 CV の Out-of-Fold 値**を当てる(自分のラベルを含まない値にする)

2 は単なる保険ではなく、XGBoost では **+0.00108** の精度向上につながった。
学習時だけ Target Encoding が当たりすぎる状態(楽観バイアス)が解消されるため。

In [13]:
def target_encode(tr_key, tr_y, va_key, prior, smooth=20.0, inner_splits=5):
    """外側foldの学習データで fit し、学習行には内側OOF値を当てる Out-of-Fold TE。"""
    # valid/test 用: 学習fold全体の統計
    agg = tr_y.groupby(tr_key).agg(["sum", "count"])
    m = (agg["sum"] + prior * smooth) / (agg["count"] + smooth)
    va_te = va_key.map(m).fillna(prior).astype("float32")

    # 学習行用: 内側CVのOOF値
    tr_te = pd.Series(np.full(len(tr_key), prior, dtype="float32"), index=tr_key.index)
    inner = StratifiedKFold(n_splits=inner_splits, shuffle=True, random_state=42)
    for i_tr, i_va in inner.split(tr_key, tr_y):
        a = tr_y.iloc[i_tr].groupby(tr_key.iloc[i_tr]).agg(["sum", "count"])
        mm = (a["sum"] + prior * smooth) / (a["count"] + smooth)
        tr_te.iloc[i_va] = tr_key.iloc[i_va].map(mm).fillna(prior).astype("float32").values
    return tr_te, va_te

#### 7-2-3 評価用の CV ループ

特徴量の作り方(`build`)を差し替えて A/B するための関数。
`build` は fold ごとに呼ばれ、学習用と検証用の特徴量を返す。

**モデル設定は本番に寄せる**(深さ5・列サンプリング 0.3・ビン数 1024・lr 0.05)。
ここは手を抜けない。素の LightGBM(深さ無制限・列サンプリングなし・lr 0.1・300本)で
同じ A/B をやると、**厳密値 TE が -0.0083 と悪化して結論が逆に出る**。
年収のような値の種類（ユニーク値）が多い列の TE は 1 値あたりの行数が少なく分散が大きいので、
正則化が効いていない木はそのノイズに飛びついてしまう。
TE が効くのは「木を弱くしてから」という順序に意味がある。

In [14]:
# 本番の設定(深さ5 / 列0.3 / ビン1024)。ここを緩めると TE の効果が逆転する。
DEMO_PARAMS = dict(n_estimators=1500, learning_rate=0.05, max_depth=5,
                   colsample_bytree=0.3, max_bin=1024)


def evaluate(build, n_splits=5, seed=42, **over):
    params = dict(DEMO_PARAMS, **over)
    oof = np.zeros(len(train))
    mask = np.zeros(len(train), dtype=bool)
    for fold, (tr, va) in enumerate(skf.split(train, y)):
        Xtr, Xva = build(tr, va)
        evaluate.last_cols = list(Xtr.columns)   # 実際に学習へ渡した列を控えておく
        model = LGBMClassifier(random_state=seed, verbosity=-1, **params)
        model.fit(Xtr, y.iloc[tr])
        oof[va] = model.predict_proba(Xva)[:, 1]
        mask[va] = True
        if fold + 1 >= n_splits:
            break
    score = roc_auc_score(y[mask], oof[mask])
    print(f"AUC: {score:.5f}  (列数 {len(evaluate.last_cols)})")
    return score


evaluate.last_cols = []
COLS_LOG = {}          # ステップ名 -> 使った列(あとで一覧表にする)


def log_cols(label, prev=None, show=8):
    """直前の evaluate() が使った列を記録し、前ステップからの増分を表示する。"""
    cols = list(evaluate.last_cols)
    COLS_LOG[label] = cols
    added = [c for c in cols if prev is None or c not in COLS_LOG.get(prev, [])]
    print(f"{chr(10)}[{label}] 合計 {len(cols)} 列 / このステップで追加 {len(added)} 列")
    if added:
        for i in range(0, min(len(added), show), 4):
            print("   ", "  ".join(f"{c:<30}" for c in added[i:i + 4]).rstrip())
        if len(added) > show:
            print(f"    … 他 {len(added) - show} 列")
    return cols

### 7-3 特徴量を1種類ずつ足して比べる(A → E)

時間短縮のため **2 fold** で比較する。数値は本番(5 fold・収束まで学習)とは揃わないが、
施策同士の比較には使える。1 ステップおよそ 2〜4 分。

| ステップ | 足すもの | 比較元 |
|---|---|---|
| A | 生の 13 列(ベースライン) | — |
| B | + 厳密値 Target Encoding(13 列) | A |
| C | + Count Encoding(13 列) | B |
| D | + 四則演算(9 列。効かない例) | C |
| E | + 交互作用 1 組の Target Encoding(1 列。本番で一部採用) | C |

#### A. ベースライン(生の特徴量のみ)

In [15]:
def prep_cat(df):
    out = df.copy()
    for c in CAT_COLS:
        cats = pd.concat([train[c], test[c]]).astype("category").cat.categories
        out[c] = pd.Categorical(out[c], categories=cats)
    return out

X_raw = prep_cat(train[ALL_COLS])

def build_base(tr, va):
    return X_raw.iloc[tr], X_raw.iloc[va]

score_base = evaluate(build_base, n_splits=2)
log_cols("A. 生の特徴量");

AUC: 0.94273  (列数 13)

[A. 生の特徴量] 合計 13 列 / このステップで追加 13 列
    Age                             Annual_Income_USD               Daily_Commute_km                Number_of_Cars_Owned
    Charging_Stations_Near_Home     Charging_Stations_Near_Work     Environmental_Concern_Level     Gender
    … 他 5 列


#### B. + 厳密値 Target Encoding(全13列)

13 列すべてを値のままキーにして、値ごとの購入率を足す(本番の Target Encoding を平滑化1種に縮めたもの)。

In [16]:
prior = y.mean()

def build_te(tr, va):
    Xtr, Xva = X_raw.iloc[tr].copy(), X_raw.iloc[va].copy()
    for c in ALL_COLS:
        key = train[c].astype(str)
        t, v = target_encode(key.iloc[tr], y.iloc[tr], key.iloc[va], prior)
        Xtr[f"te_{c}"], Xva[f"te_{c}"] = t.values, v.values
    return Xtr, Xva

score_te = evaluate(build_te, n_splits=2)
print(f"ベースラインとの差: {score_te - score_base:+.5f}")
log_cols("B. + 厳密値TE", prev="A. 生の特徴量");

AUC: 0.94450  (列数 26)
ベースラインとの差: +0.00177

[B. + 厳密値TE] 合計 26 列 / このステップで追加 13 列
    te_Age                          te_Annual_Income_USD            te_Daily_Commute_km             te_Number_of_Cars_Owned
    te_Charging_Stations_Near_Home  te_Charging_Stations_Near_Work  te_Environmental_Concern_Level  te_Gender
    … 他 5 列


#### C. + Count Encoding

値の出現頻度。**目的変数を使わないので train+test をまとめて数えてよい**(リークしない)。
TE とは別の情報なので加算的に効く。

In [17]:
count_maps = {c: pd.concat([train[c], test[c]]).astype(str).value_counts() for c in ALL_COLS}

def build_te_cnt(tr, va):
    Xtr, Xva = build_te(tr, va)
    for c in ALL_COLS:
        key = train[c].astype(str)
        Xtr[f"cnt_{c}"] = key.iloc[tr].map(count_maps[c]).values
        Xva[f"cnt_{c}"] = key.iloc[va].map(count_maps[c]).values
    return Xtr, Xva

score_te_cnt = evaluate(build_te_cnt, n_splits=2)
print(f"TE のみとの差: {score_te_cnt - score_te:+.5f}")
log_cols("C. + Count Encoding", prev="B. + 厳密値TE");

AUC: 0.94479  (列数 39)
TE のみとの差: +0.00029

[C. + Count Encoding] 合計 39 列 / このステップで追加 13 列
    cnt_Age                         cnt_Annual_Income_USD           cnt_Daily_Commute_km            cnt_Number_of_Cars_Owned
    cnt_Charging_Stations_Near_Home  cnt_Charging_Stations_Near_Work  cnt_Environmental_Concern_Level  cnt_Gender
    … 他 5 列


#### D. + 四則演算(効かない例)

「充電スタンド数の自宅+職場」「年収÷通勤距離」など直感的な合成指標。
木は 1 列ずつしか分割できないので効きそうに見えるが、**合成データの生成過程に
そうした関係がない**ため効かない。どのモデルでも無効〜悪化だった(03 の 8-2)。

In [18]:
PAIRS = [("Charging_Stations_Near_Home", "Charging_Stations_Near_Work"),
         ("Annual_Income_USD", "Daily_Commute_km"),
         ("Age", "Annual_Income_USD")]

def build_arith(tr, va):
    Xtr, Xva = build_te_cnt(tr, va)
    for a, b in PAIRS:
        for X_, idx in ((Xtr, tr), (Xva, va)):
            X_[f"{a}_sum_{b}"] = train[a].iloc[idx].values + train[b].iloc[idx].values
            X_[f"{a}_diff_{b}"] = train[a].iloc[idx].values - train[b].iloc[idx].values
            X_[f"{a}_ratio_{b}"] = train[a].iloc[idx].values / (train[b].iloc[idx].values + 1e-6)
    return Xtr, Xva

score_arith = evaluate(build_arith, n_splits=2)
print(f"TE+Count との差: {score_arith - score_te_cnt:+.5f}")
log_cols("D. + 四則演算", prev="C. + Count Encoding");

AUC: 0.94476  (列数 48)
TE+Count との差: -0.00003

[D. + 四則演算] 合計 48 列 / このステップで追加 9 列
    Charging_Stations_Near_Home_sum_Charging_Stations_Near_Work  Charging_Stations_Near_Home_diff_Charging_Stations_Near_Work  Charging_Stations_Near_Home_ratio_Charging_Stations_Near_Work  Annual_Income_USD_sum_Daily_Commute_km
    Annual_Income_USD_diff_Daily_Commute_km  Annual_Income_USD_ratio_Daily_Commute_km  Age_sum_Annual_Income_USD       Age_diff_Annual_Income_USD
    … 他 1 列


#### E. + 交互作用: 自宅充電の可否 × 自宅スタンド数(本番で一部採用)

EDA で見つかった唯一の交互作用。「自宅で充電できない人だけ、近くのスタンドが多いほど買う」という関係がある。
C(Target Encoding + Count)に、この組み合わせをキーにした Target Encoding を **1 列だけ**足す。

本番では LightGBM・RealMLP に採用している(03 の 3-3・5-8)。効果は誤差の範囲で、
木は「自宅充電で分けてからスタンド数を見る」という2段の分割で、この関係を自力で表現できる。

In [19]:
def build_pair(tr, va):
    Xtr, Xva = build_te_cnt(tr, va)
    # 2 列の値を文字列でつないだ組み合わせキー(自宅充電の可否 × 自宅スタンド数。30 種類)
    key = train["Home_Charging_Possible"].astype(str) + "_" + train["Charging_Stations_Near_Home"].astype(str)
    t, v = target_encode(key.iloc[tr], y.iloc[tr], key.iloc[va], prior)
    name = "te_Home_Charging_PossibleXCharging_Stations_Near_Home"
    Xtr[name], Xva[name] = t.values, v.values
    return Xtr, Xva

score_pair = evaluate(build_pair, n_splits=2)
print(f"TE+Count との差: {score_pair - score_te_cnt:+.5f}")
log_cols("E. + 交互作用(1組)", prev="C. + Count Encoding");

AUC: 0.94482  (列数 40)
TE+Count との差: +0.00002

[E. + 交互作用(1組)] 合計 40 列 / このステップで追加 1 列
    te_Home_Charging_PossibleXCharging_Stations_Near_Home


#### A → E のまとめ

D と E は、どちらも C(Target Encoding + Count)に足した場合との比較。

In [20]:
LABELS = ["A. 生の特徴量", "B. + 厳密値TE", "C. + Count Encoding", "D. + 四則演算", "E. + 交互作用(1組)"]
PREV = {"B. + 厳密値TE": "A. 生の特徴量", "C. + Count Encoding": "B. + 厳密値TE",
        "D. + 四則演算": "C. + Count Encoding", "E. + 交互作用(1組)": "C. + Count Encoding"}
SCORES = dict(zip(LABELS, [score_base, score_te, score_te_cnt, score_arith, score_pair]))
pd.DataFrame({
    "AUC": [SCORES[l] for l in LABELS],
    "比較元": [PREV.get(l, "—") for l in LABELS],
    "比較元との差": [SCORES[l] - SCORES[PREV[l]] if l in PREV else np.nan for l in LABELS],
    "列数": [len(COLS_LOG[l]) for l in LABELS],
}, index=LABELS).round(5)

,AUC,比較元,比較元との差,列数
A. 生の特徴量,0.94273,—,NaN,13
B. + 厳密値TE,0.94450,A. 生の特徴量,0.00177,26
C. + Count Encoding,0.94479,B. + 厳密値TE,0.00029,39
D. + 四則演算,0.94476,C. + Count Encoding,-0.00003,48
E. + 交互作用(1組),0.94482,C. + Count Encoding,0.00002,40


#### この比較で作った列の全一覧

上の A〜E で何が増えたのかを、列名まで展開して確認する。
**四則演算(D)は 9 列も増えるのに AUC が動かない**ことが目で見て分かる。

In [21]:
for label in LABELS:
    cols = COLS_LOG[label]
    prev = PREV.get(label)
    added = [c for c in cols if prev is None or c not in COLS_LOG[prev]]
    print(f"\n■ {label}  —  合計 {len(cols)} 列 / 追加 {len(added)} 列(比較元: {prev or 'なし'})")
    for i in range(0, len(added), 3):
        print("    " + "  ".join(f"{c:<38}" for c in added[i:i + 3]).rstrip())


■ A. 生の特徴量  —  合計 13 列 / 追加 13 列(比較元: なし)
    Age                                     Annual_Income_USD                       Daily_Commute_km
    Number_of_Cars_Owned                    Charging_Stations_Near_Home             Charging_Stations_Near_Work
    Environmental_Concern_Level             Gender                                  City_Type
    Current_Car_Type                        Home_Charging_Possible                  Subsidy_Available
    Range_Anxiety_Level

■ B. + 厳密値TE  —  合計 26 列 / 追加 13 列(比較元: A. 生の特徴量)
    te_Age                                  te_Annual_Income_USD                    te_Daily_Commute_km
    te_Number_of_Cars_Owned                 te_Charging_Stations_Near_Home          te_Charging_Stations_Near_Work
    te_Environmental_Concern_Level          te_Gender                               te_City_Type
    te_Current_Car_Type                     te_Home_Charging_Possible               te_Subsidy_Available
    te_Range_Anxiety_Level

■ C. + Count Encoding  —